In [1]:
# CELL 1 — Install dependencies for Google Colab
# Colab normally provides a compatible torch/torchvision build. Do not reinstall
# them from a separate CUDA wheel index unless you have a specific compatibility need.
import subprocess, sys

def pip(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages.split())

pip('scikit-learn matplotlib seaborn pandas Pillow tqdm opencv-python scikit-image')
import torch, torchvision
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {g.name} | VRAM: {g.total_memory/1e9:.1f}GB')
else:
    print('WARNING: GPU not detected. In Colab, choose Runtime → Change runtime type → GPU.')
print('Dependencies ready!')


PyTorch: 2.7.1+cu118
Torchvision: 0.22.1+cu118
GPU: NVIDIA GeForce RTX 3050 4GB Laptop GPU | VRAM: 4.3GB
Dependencies ready!


In [2]:
# CELL 2 — Imports
import os, random, warnings, time, re
import cv2
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.transforms as T
import torchvision.models as tvm

from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from skimage.filters import threshold_otsu

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Cell 2 OK!')

Device: cuda
Cell 2 OK!


In [3]:
# CELL 3 — Google Colab Configuration
import os, glob

# Mount Google Drive so the dataset and result files persist across Colab sessions.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Automatically locate the folder containing the 8 selected MVTec AD categories.
# Recommended Drive layout: MyDrive/dataset/dataset/<category>/train/good, test/...
ALL_CATS = [
    'bottle', 'carpet', 'hazelnut', 'metal_nut',
    'wood', 'leather', 'cable', 'transistor'
]

CATEGORY_SET = set(ALL_CATS)

def _is_mvtec_root(path):
    if not os.path.isdir(path):
        return False
    present = {d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))}
    return CATEGORY_SET.issubset(present)

if IN_COLAB:
    preferred = [
        '/content/drive/MyDrive/dataset/dataset',
        '/content/drive/MyDrive/dataset',
        '/content/drive/MyDrive/mvtec_anomaly_detection',
        '/content/drive/MyDrive/datasets/mvtec_anomaly_detection',
        '/content/drive/MyDrive/Datasets/mvtec_anomaly_detection',
        '/content/mvtec_anomaly_detection',
    ]
else:
    preferred = ['./dataset/dataset', './dataset', './mvtec_anomaly_detection']

DATASET_PATH = next((p for p in preferred if _is_mvtec_root(p)), None)
if DATASET_PATH is None:
    search_roots = ['/content/drive/MyDrive'] if IN_COLAB else ['.']
    for search_root in search_roots:
        for root, dirs, files in os.walk(search_root):
            depth = os.path.relpath(root, search_root).count(os.sep)
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in {'AMEM_Net_V6_Colab', '__pycache__'}]
            if depth > 6:
                dirs[:] = []
                continue
            if _is_mvtec_root(root):
                DATASET_PATH = root
                break
        if DATASET_PATH:
            break

if DATASET_PATH is None:
    raise FileNotFoundError(
        'Could not find a dataset root containing all 8 selected category folders. '
        'In Google Drive, put the extracted category folders under My Drive/dataset/dataset/, '
        'or set DATASET_PATH manually to the folder that directly contains the category folders.'
    )

missing_structure = []
for cat in ALL_CATS:
    for rel in [('train','good'), ('test',)]:
        p = os.path.join(DATASET_PATH, cat, *rel)
        if not os.path.isdir(p):
            missing_structure.append(p)
if missing_structure:
    raise FileNotFoundError(
        'Dataset root was found, but expected MVTec folders are missing. First missing paths: '
        + ', '.join(missing_structure[:8])
    )
print('DATASET PATH VERIFIED:', DATASET_PATH)
print('Verified categories:', ', '.join(ALL_CATS))

IMG_SIZE    = 128
BATCH_SIZE  = 2  # conservative for a 4GB GPU; increase to 4 only if memory allows
NUM_WORKERS = 0

# 50 epochs as requested.
EPOCHS      = 50

# ---------- AMEM-Net V6 ----------
N_EXPERTS = 4

# V2 used only 100 slots/expert. V3 keeps a much richer normal patch memory.
# 4 × 2048 = 8192 normal patch representatives in total.
EXPERT_MEMORY_SLOTS = 512  # reduced memory footprint for a 4GB GPU laptop

# Number of normal training patches used to build each expert memory.
# The memory contains actual normal feature vectors, not synthetic/fabricated scores.
MAX_BANK_PATCHES = 20000  # caps AMEM normal feature bank to reduce RAM/VRAM pressure

# Router / self-supervised training.
ROUTER_LR = 2e-4
ROUTER_WEIGHT_DECAY = 1e-4
PSEUDO_MARGIN = 0.08
NORMAL_WEIGHT = 1.0
MARGIN_WEIGHT = 1.0
ROUTER_KL_WEIGHT = 0.5
# Number of memory representatives used during router training.
# This is the V3 training-time memory cap; final expert memories remain unchanged.
ROUTER_MEMORY_PER_EXPERT = 32  # small router-training sample for laptop memory
IDENTITY_WEIGHT = 0.05

# Score construction.
TOP_PATCH_FRAC = 0.05
FEEDBACK_TEMP = 0.10

# Validation threshold search uses only held-out normal validation images plus
# self-supervised pseudo-anomalies generated from those normal validation images.
THRESH_Q = 0.995

# Existing baselines
ENTROPY_W   = 0.0001
MEM_SHRINK  = 0.0025
LR          = 1e-4

# PatchCore settings
PC_RATIO = 0.25
PC_K = 9

OUTPUT_ROOT = os.path.abspath('./AMEM_Net_V6_Laptop')  # local Jupyter output; no Google Drive dependency
SAVE_DIR  = os.path.join(OUTPUT_ROOT, 'results_v6_wrn50_youden_8cats')
MODEL_DIR = os.path.join(OUTPUT_ROOT, 'saved_models_v6_wrn50_youden')
CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, 'checkpoints_v6_wrn50_youden')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Selected category count:', len(ALL_CATS))
print('Selected categories:', ', '.join(ALL_CATS))
print('Checkpoint directory:', CHECKPOINT_DIR)
print('Results directory (8 categories only):', SAVE_DIR)
print('Resume mode: ON — per-epoch checkpoints and per-model result persistence enabled.')
print('Laptop profile: batch size 2, 512 AMEM slots/expert, max 20,000 AMEM bank patches.')

# SAFETY: do not delete or overwrite previous experiment files.
# This fixed copy writes to separate folders.
print('Safety mode: V3/V4/V5 results and models are preserved; V6 writes to separate folders.')

MEAN_T = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD_T  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
def denorm(t):
    return (t.cpu()*STD_T+MEAN_T).clamp(0,1)

MEAN_D = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(DEVICE)
STD_D  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(DEVICE)
def to_01(t):
    return (t.to(DEVICE)*STD_D + MEAN_D).clamp(0,1)

print('Dataset   :', DATASET_PATH)
print('IMG_SIZE  :', IMG_SIZE)
print('BATCH_SIZE:', BATCH_SIZE)
print('EPOCHS    :', EPOCHS)
print('AMEM      : WideResNet50-2,', N_EXPERTS, 'experts ×', EXPERT_MEMORY_SLOTS, 'patch slots')
print('Device    :', DEVICE)

if not os.path.exists(DATASET_PATH):
    print('ERROR: Dataset not found:', DATASET_PATH)
else:
    found = [c for c in ALL_CATS if os.path.exists(os.path.join(DATASET_PATH,c))]
    print('Found categories:', len(found), '/', len(ALL_CATS))
print('CELL 3 OK')

DATASET PATH VERIFIED: .\Downloads\mvtec_anomaly_detection
Verified categories: bottle, carpet, hazelnut, metal_nut, wood, leather, cable, transistor
Selected category count: 8
Selected categories: bottle, carpet, hazelnut, metal_nut, wood, leather, cable, transistor
Checkpoint directory: C:\Users\chigu\AMEM_Net_V6_Laptop\checkpoints_v6_wrn50_youden
Results directory (8 categories only): C:\Users\chigu\AMEM_Net_V6_Laptop\results_v6_wrn50_youden_8cats
Resume mode: ON — per-epoch checkpoints and per-model result persistence enabled.
Laptop profile: batch size 2, 512 AMEM slots/expert, max 20,000 AMEM bank patches.
Safety mode: V3/V4/V5 results and models are preserved; V6 writes to separate folders.
Dataset   : .\Downloads\mvtec_anomaly_detection
IMG_SIZE  : 128
BATCH_SIZE: 2
EPOCHS    : 50
AMEM      : WideResNet50-2, 4 experts × 512 patch slots
Device    : cuda
Found categories: 8 / 8
CELL 3 OK


In [4]:
# CELL 4 — PatchCore: 8 Selected Categories (leakage-safe evaluation)
import re  # needed to normalize duplicate filename suffixes such as "(1)"
# WideResNet50-2 is retained ONLY as the PatchCore baseline.
# Threshold is calibrated from held-out normal validation images, never from test labels.

class MVTecDataset(Dataset):
    def __init__(self, root, cat, split='train', size=128):
        self.size = size; self.data=[]
        self.tf=T.Compose([T.Resize((size,size)),T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        if split=='train':
            gd=os.path.join(root,cat,'train','good')
            for f in sorted(os.listdir(gd)):
                if f.lower().endswith(('.png','.jpg','.bmp')): self.data.append((os.path.join(gd,f),0))
        else:
            td=os.path.join(root,cat,'test')
            for d in sorted(os.listdir(td)):
                dp=os.path.join(td,d)
                if not os.path.isdir(dp): continue
                lbl=0 if d=='good' else 1
                for f in sorted(os.listdir(dp)):
                    if f.lower().endswith(('.png','.jpg','.bmp')): self.data.append((os.path.join(dp,f),lbl))
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        img=Image.open(self.data[i][0]).convert('RGB')
        return self.tf(img),self.data[i][1]

def normal_quantile_threshold(val_scores, q=THRESH_Q):
    v=np.asarray(val_scores,dtype=np.float64)
    v=v[np.isfinite(v)]
    return float(np.quantile(v,q))

def metrics_at_threshold(scores, labels, threshold):
    sc=np.asarray(scores,dtype=np.float64); lb=np.asarray(labels,dtype=np.int32)
    sn=(sc-sc.min())/(sc.max()-sc.min()+1e-12)
    auroc=roc_auc_score(lb,sn)*100 if len(np.unique(lb))==2 else float('nan')
    aucpr=average_precision_score(lb,sn)*100 if len(np.unique(lb))==2 else float('nan')
    # threshold is defined in original score units
    pred=(sc>=threshold).astype(int)
    f1=f1_score(lb,pred,zero_division=0)*100
    prec=precision_score(lb,pred,zero_division=0)*100
    rec=recall_score(lb,pred,zero_division=0)*100
    acc=accuracy_score(lb,pred)*100
    tn=int(((pred==0)&(lb==0)).sum()); fp=int(((pred==1)&(lb==0)).sum())
    fpr_value=100.0*fp/max(1,tn+fp)
    fpr_c,tpr_c,_=roc_curve(lb,sn) if len(np.unique(lb))==2 else (np.array([0,1]),np.array([0,1]),None)
    return dict(AUROC=round(auroc,2),AUCPR=round(aucpr,2),F1=round(f1,2),
                Precision=round(prec,2),Recall=round(rec,2),Accuracy=round(acc,2),FPR=round(fpr_value,2),
                threshold=float(threshold),fpr_c=fpr_c,tpr_c=tpr_c,sn=sn,lb=lb)


# Shared helper used by MemAE training in Cell 5.
def entropy_loss(a):
    return (-a*torch.log(a+1e-12)).sum(-1).mean()

print('Loading WideResNet50-2...')
_bb=tvm.wide_resnet50_2(weights=tvm.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(DEVICE).eval()
_feats={}
def _h2(m,i,o): _feats['l2']=o
def _h3(m,i,o): _feats['l3']=o
_bb.layer2.register_forward_hook(_h2); _bb.layer3.register_forward_hook(_h3)
for p in _bb.parameters(): p.requires_grad=False
print('WideResNet50-2 ready!')

@torch.no_grad()
def extract_patches(loader):
    out=[]
    for imgs,_ in tqdm(loader,leave=False,desc='  extracting'):
        _bb(imgs.to(DEVICE)); f2=_feats['l2']; f3=_feats['l3']
        f3u=F.interpolate(f3,size=f2.shape[-2:],mode='bilinear',align_corners=False)
        f=torch.cat([f2,f3u],1); B,C,H,W=f.shape
        out.append(f.permute(0,2,3,1).reshape(-1,C).cpu().numpy())
    return np.vstack(out)

@torch.no_grad()
def score_loader(loader,nn_model):
    scores=[]; labels=[]; maps=[]
    for imgs,lbls in tqdm(loader,leave=False,desc='  scoring'):
        _bb(imgs.to(DEVICE)); f2=_feats['l2']; f3=_feats['l3']
        f3u=F.interpolate(f3,size=f2.shape[-2:],mode='bilinear',align_corners=False)
        f=torch.cat([f2,f3u],1); B,C,H,W=f.shape
        pf=f.permute(0,2,3,1).reshape(B,H*W,C).cpu().numpy()
        for j in range(B):
            d,_=nn_model.kneighbors(pf[j].astype(np.float32))
            dmap=d.mean(1).reshape(H,W)
            scores.append(float(dmap.max())); labels.append(int(lbls[j])); maps.append(dmap)
    return np.array(scores),np.array(labels),maps


def load_gt_mask(img_path, cat, size=IMG_SIZE):
    """Load the matching MVTec pixel mask. Handles duplicate-name suffixes like '(1)'."""
    defect=os.path.basename(os.path.dirname(img_path))
    if defect == 'good':
        return np.zeros((size,size),dtype=np.uint8)

    stem=os.path.splitext(os.path.basename(img_path))[0]
    gt_dir=os.path.join(DATASET_PATH,cat,'ground_truth',defect)

    # Standard MVTec naming: 003.png -> 003_mask.png.
    candidates=[os.path.join(gt_dir, stem+'_mask.png')]

    # Some uploads/OS file managers rename duplicate images to e.g. '003 (1).png'.
    # Match the original stem ('003') to its standard ground-truth mask.
    clean_stem=re.sub(r'\s*\(\d+\)$', '', stem).strip()
    if clean_stem != stem:
        candidates.append(os.path.join(gt_dir, clean_stem+'_mask.png'))

    # Also try common image extensions for the mask, preserving the required *_mask stem.
    for candidate_stem in dict.fromkeys([stem, clean_stem]):
        for ext in ('.png', '.PNG'):
            p=os.path.join(gt_dir, candidate_stem+'_mask'+ext)
            if p not in candidates:
                candidates.append(p)

    mask_path=next((p for p in candidates if os.path.isfile(p)), None)
    if mask_path is None:
        available=[]
        if os.path.isdir(gt_dir):
            available=sorted(os.listdir(gt_dir))[:12]
        raise FileNotFoundError(
            f'Ground-truth mask not found for image: {img_path}\n'
            f'Expected a mask in: {gt_dir}\n'
            f'Tried stems: {stem!r} and {clean_stem!r}\n'
            f'Example files in mask folder: {available}\n'
            'Check that the dataset is the original MVTec AD structure and that test images '
            'were not renamed independently from their ground-truth masks.'
        )

    mask=Image.open(mask_path).convert('L').resize((size,size),Image.Resampling.NEAREST)
    return (np.asarray(mask)>0).astype(np.uint8)


def _maps_to_2d(maps):
    out=[]
    for m in maps:
        if torch.is_tensor(m): m=m.detach().cpu().numpy()
        a=np.asarray(m).squeeze()
        if a.ndim==1:
            side=int(round(np.sqrt(a.size)))
            if side*side != a.size: raise ValueError(f'Cannot reshape anomaly map with {a.size} values into a square map.')
            a=a.reshape(side,side)
        if a.ndim != 2: raise ValueError(f'Expected a 2D anomaly map, got shape {a.shape}')
        a=cv2.resize(a.astype(np.float32),(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
        out.append(a)
    return out


def pixel_metrics_for_category(cat, image_paths, anomaly_maps, val_normal_maps):
    """Pixel AUROC/AUPR, PRO (FPR<=0.30), and IoU using GT masks.

    IoU threshold is the 99th percentile of held-out normal validation map values;
    test masks are used only for final metric evaluation, never threshold selection.
    """
    maps=_maps_to_2d(anomaly_maps)
    valmaps=_maps_to_2d(val_normal_maps)
    masks=[load_gt_mask(p,cat) for p in image_paths]
    if not maps or len(maps)!=len(masks):
        raise ValueError(f'{cat}: anomaly maps/masks length mismatch ({len(maps)} vs {len(masks)}).')
    y=np.concatenate([m.reshape(-1) for m in masks]).astype(np.uint8)
    x=np.concatenate([a.reshape(-1) for a in maps]).astype(np.float32)
    if len(np.unique(y))==2:
        px_auc=roc_auc_score(y,x)*100.0
        px_ap=average_precision_score(y,x)*100.0
    else:
        px_auc=float('nan'); px_ap=float('nan')
    val_pixels=np.concatenate([m.reshape(-1) for m in valmaps]).astype(np.float32)
    px_thr=float(np.quantile(val_pixels,0.99))
    pred=(x>=px_thr).astype(np.uint8)
    inter=int(((pred==1)&(y==1)).sum()); union=int(((pred==1)|(y==1)).sum())
    px_iou=100.0*inter/max(1,union)

    # PRO curve: mean overlap over connected GT defect regions versus pixel FPR.
    from scipy import ndimage
    region_masks=[]; normal_pixels=[]
    for gt, amap in zip(masks,maps):
        if gt.any():
            lab,nlab=ndimage.label(gt)
            for rid in range(1,nlab+1):
                region_masks.append(lab==rid)
        else:
            normal_pixels.append(amap.reshape(-1))
    if region_masks and normal_pixels:
        norm=np.concatenate(normal_pixels)
        all_scores=np.concatenate([a.reshape(-1) for a in maps])
        thresholds=np.unique(np.quantile(all_scores,np.linspace(0.0,1.0,100)))
        fprs=[]; pros=[]
        for th in thresholds:
            overlaps=[]
            for gt,amap in zip(masks,maps):
                if gt.any():
                    lab,nlab=ndimage.label(gt)
                    pb=amap>=th
                    for rid in range(1,nlab+1):
                        reg=(lab==rid)
                        overlaps.append(float((pb&reg).sum())/max(1,int(reg.sum())))
            fpr=float((norm>=th).mean())
            fprs.append(fpr); pros.append(float(np.mean(overlaps)) if overlaps else 0.0)
        order=np.argsort(fprs); fx=np.asarray(fprs)[order]; py=np.asarray(pros)[order]
        # Collapse duplicate FPR points by maximum PRO and integrate only 0..0.30.
        ux=[]; uy=[]
        for f in np.unique(fx): ux.append(float(f)); uy.append(float(np.max(py[fx==f])))
        ux=np.asarray(ux); uy=np.asarray(uy)
        if ux[0] > 0: ux=np.insert(ux,0,0.0); uy=np.insert(uy,0,uy[0])
        cutoff=0.30
        keep=ux < cutoff
        xx=ux[keep]; yy=uy[keep]
        if len(xx)==0 or xx[-1] < cutoff:
            ycut=float(np.interp(cutoff,ux,uy)) if len(ux)>1 else float(uy[0])
            xx=np.append(xx,cutoff); yy=np.append(yy,ycut)
        aupro=100.0*float(np.trapezoid(yy,xx))/cutoff
    else:
        aupro=float('nan')
    return {'Pixel-AUROC':round(float(px_auc),2),'Pixel-AUPR':round(float(px_ap),2),
            'AUPRO':round(float(aupro),2),'Pixel-IoU':round(float(px_iou),2),'PixelThreshold':px_thr}

PC_CSV = f'{SAVE_DIR}/patchcore_results.csv'
if os.path.isfile(PC_CSV):
    try:
        _pc_old = pd.read_csv(PC_CSV)
    except Exception:
        _pc_old = pd.DataFrame()
else:
    _pc_old = pd.DataFrame()
if not _pc_old.empty and 'Category' in _pc_old.columns:
    _pc_old = _pc_old[_pc_old['Category'].astype(str).isin(ALL_CATS)].copy()
pc_results = _pc_old.to_dict('records') if not _pc_old.empty else []
pc_roc = {}
_pc_done = set(_pc_old['Category'].astype(str)) if (not _pc_old.empty and 'Category' in _pc_old.columns) else set()
for cat in ALL_CATS:
    if cat in _pc_done:
        print(f'[PATCHCORE: {cat.upper()}] already in saved results; skipping completed category.')
        continue
    if not os.path.exists(os.path.join(DATASET_PATH,cat)): continue
    print(f'\n[PATCHCORE: {cat.upper()}]')
    full=MVTecDataset(DATASET_PATH,cat,'train',IMG_SIZE)
    nval=max(1,int(0.1*len(full))); ntr=len(full)-nval
    tr_ds,val_ds=random_split(full,[ntr,nval],generator=torch.Generator().manual_seed(SEED))
    te_ds=MVTecDataset(DATASET_PATH,cat,'test',IMG_SIZE)
    tr_ld=DataLoader(tr_ds,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    val_ld=DataLoader(val_ds,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    te_ld=DataLoader(te_ds,BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    # Persist PatchCore's fitted normal feature memory per category so it can be reused.
    pc_mem_path = os.path.join(MODEL_DIR, f'patchcore_memory_{cat}.npz')
    if os.path.isfile(pc_mem_path):
        try:
            with np.load(pc_mem_path) as _z:
                mem = _z['memory'].astype(np.float32)
            print(f'  Loaded saved PatchCore memory bank: {pc_mem_path}')
        except Exception as _e:
            print(f'  Could not load PatchCore memory bank ({_e}); rebuilding it.')
            mem = extract_patches(tr_ld)
    else:
        mem = extract_patches(tr_ld)
    # Apply the same deterministic coreset sampling only to a newly extracted full bank.
    if len(mem) > max(300, int(len(mem)*PC_RATIO)) and not os.path.isfile(pc_mem_path):
        n_keep=max(300,int(len(mem)*PC_RATIO))
        idx=np.random.default_rng(SEED).choice(len(mem),min(n_keep,len(mem)),replace=False)
        mem=mem[idx]
    nn_m=NearestNeighbors(n_neighbors=min(PC_K, len(mem)),metric='euclidean',n_jobs=-1).fit(mem)
    val_sc,_,val_maps=score_loader(val_ld,nn_m); threshold=normal_quantile_threshold(val_sc)
    # Cache features and threshold after validation calibration; safe to reuse on a rerun.
    try:
        np.savez_compressed(pc_mem_path, memory=mem.astype(np.float32), threshold=np.asarray(threshold,dtype=np.float32))
        print(f'  Saved PatchCore memory bank: {pc_mem_path}')
    except Exception as _e:
        print(f'  WARNING: PatchCore memory bank cache could not be saved: {_e}')
    sc,lb,test_maps=score_loader(te_ld,nn_m); mt=metrics_at_threshold(sc,lb,threshold)
    val_maps_normal=[m for m,lbl in zip(val_maps, [0]*len(val_maps))]
    pix= pixel_metrics_for_category(cat, [p for p,_ in te_ds.data], test_maps, val_maps_normal)
    pc_roc[cat]={'fpr':mt['fpr_c'],'tpr':mt['tpr_c'],'auroc':mt['AUROC']}
    pc_results.append({'Category':cat,'Model':'PatchCore','AUROC':mt['AUROC'],'AUC-PR':mt['AUCPR'],
        'Accuracy':mt['Accuracy'],'Precision':mt['Precision'],'Recall':mt['Recall'],'F1':mt['F1'],**{k:v for k,v in pix.items() if k not in ('Pixel-IoU','PixelThreshold')}})
    print(f"  AUROC={mt['AUROC']:.2f}% F1={mt['F1']:.2f}% Prec={mt['Precision']:.2f}% Rec={mt['Recall']:.2f}% Acc={mt['Accuracy']:.2f}%")
    _pc_save = PC_CSV + '.tmp'
    pd.DataFrame(pc_results).drop(columns=['FPR','Pixel-IoU','PixelThreshold'], errors='ignore').to_csv(_pc_save,index=False)
    os.replace(_pc_save, PC_CSV)

df_pc=pd.DataFrame(pc_results)
print('\nPatchCore complete. PatchCore thresholds were calibrated on held-out normal validation images only.')


Loading WideResNet50-2...
WideResNet50-2 ready!

[PATCHCORE: BOTTLE]


  Saved PatchCore memory bank: C:\Users\chigu\AMEM_Net_V6_Laptop\saved_models_v6_wrn50_youden\patchcore_memory_bottle.npz


  AUROC=99.60% F1=98.44% Prec=96.92% Rec=100.00% Acc=97.59%

[PATCHCORE: CARPET]


  Saved PatchCore memory bank: C:\Users\chigu\AMEM_Net_V6_Laptop\saved_models_v6_wrn50_youden\patchcore_memory_carpet.npz


  AUROC=98.52% F1=97.73% Prec=98.85% Rec=96.63% Acc=96.58%

[PATCHCORE: HAZELNUT]


  Saved PatchCore memory bank: C:\Users\chigu\AMEM_Net_V6_Laptop\saved_models_v6_wrn50_youden\patchcore_memory_hazelnut.npz


... (7 outputs left out)

  Saved PatchCore memory bank: C:\Users\chigu\AMEM_Net_V6_Laptop\saved_models_v6_wrn50_youden\patchcore_memory_cable.npz


  AUROC=91.25% F1=85.87% Prec=85.87% Rec=85.87% Acc=82.67%

[PATCHCORE: TRANSISTOR]


  Saved PatchCore memory bank: C:\Users\chigu\AMEM_Net_V6_Laptop\saved_models_v6_wrn50_youden\patchcore_memory_transistor.npz


  AUROC=92.71% F1=80.52% Prec=83.78% Rec=77.50% Acc=85.00%

PatchCore complete. PatchCore thresholds were calibrated on held-out normal validation images only.


In [11]:
# CELL 5 — CNN-AE | ResNet18-AE | MemAE | AMEM-Net with WideResNet50-2
#
# AMEM-Net V6 design:
#   Frozen WideResNet50-2 multi-scale encoder
#             ↓
#   normal image-mode clustering
#             ↓
#   4 specialized dense normal patch memories
#             ↓
#   trainable dynamic router
#             ↓
#   self-supervised pseudo-anomaly margin training
#             ↓
#   memory reconstruction-error feedback
#             ↓
#   top-patch anomaly score
#
# Key difference from V2:
#   V2 compressed each expert to only 100 K-Means prototypes and trained the
#   router only to imitate low-error expert weights.
#
#   V3 retains 8192 normal patch representatives in total and trains the
#   router using an explicit normal-vs-pseudo-anomaly separation objective.
#
# No test labels are used anywhere in AMEM training or threshold selection.

import time
import math
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize as sk_normalize
from torch.cuda.amp import GradScaler, autocast

# ---------- robust checkpoint helpers ----------
def _atomic_torch_save(obj, path):
    """Write to a temporary file first to reduce the chance of a partial Drive checkpoint."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp_path = path + '.tmp'
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)

def _load_checkpoint(path):
    """Load full training state across PyTorch versions."""
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:  # older PyTorch without weights_only argument
        return torch.load(path, map_location=DEVICE)

def _rng_checkpoint_state():
    state = {'torch_rng': torch.get_rng_state()}
    if torch.cuda.is_available():
        state['cuda_rng_all'] = torch.cuda.get_rng_state_all()
    return state

def _restore_rng_state(state):
    if not state:
        return
    try:
        if state.get('torch_rng') is not None:
            torch.set_rng_state(state['torch_rng'].cpu())
        if torch.cuda.is_available() and state.get('cuda_rng_all') is not None:
            torch.cuda.set_rng_state_all(state['cuda_rng_all'])
    except Exception as exc:
        print('Warning: could not fully restore RNG state:', exc)

def _checkpoint_path(model_type, category):
    return os.path.join(CHECKPOINT_DIR, f'{model_type}_{category}_last.pt')

def _upsert_result(rows, new_row):
    """Keep one current measured row per Category/Model pair."""
    key = (str(new_row.get('Category')), str(new_row.get('Model')))
    rows[:] = [r for r in rows if (str(r.get('Category')), str(r.get('Model'))) != key]
    rows.append(new_row)
    return rows

# ---------- loaders ----------
def get_loaders(cat):
    train_tf = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(0.5),
        T.RandomRotation(5),
        T.ColorJitter(0.08, 0.08, 0.04),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],
                    [0.229,0.224,0.225])
    ])

    eval_tf = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],
                    [0.229,0.224,0.225])
    ])

    # Separate dataset objects are used so the held-out validation subset
    # never receives random training augmentation.
    raw = MVTecDataset(DATASET_PATH, cat, 'train', IMG_SIZE)

    vn = max(1, int(0.10 * len(raw)))
    tn = len(raw) - vn

    g = torch.Generator().manual_seed(SEED)
    perm = torch.randperm(len(raw), generator=g).tolist()

    tr_idx = perm[:tn]
    vl_idx = perm[tn:]

    train_base = MVTecDataset(DATASET_PATH, cat, 'train', IMG_SIZE)
    train_base.tf = train_tf

    val_base = MVTecDataset(DATASET_PATH, cat, 'train', IMG_SIZE)
    val_base.tf = eval_tf

    tr_s = Subset(train_base, tr_idx)
    vl_s = Subset(val_base, vl_idx)

    te_ds = MVTecDataset(DATASET_PATH, cat, 'test', IMG_SIZE)
    te_ds.tf = eval_tf

    kw = dict(
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda')
    )

    trl = DataLoader(tr_s, BATCH_SIZE, shuffle=True, **kw)
    vll = DataLoader(vl_s, BATCH_SIZE, shuffle=False, **kw)
    tel = DataLoader(te_ds, BATCH_SIZE, shuffle=False, **kw)

    n1 = sum(1 for _, l in te_ds.data if l == 1)
    return trl, vll, tel, n1


# ============================================================
# COMMON AUTOENCODER BASELINES
# ============================================================

class CB(nn.Module):
    def __init__(self, i, o, s=1):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(i, o, 3, s, 1, bias=False),
            nn.BatchNorm2d(o),
            nn.LeakyReLU(0.2, True)
        )
    def forward(self, x):
        return self.b(x)

class DB(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.ConvTranspose2d(i, o, 4, 2, 1, bias=False),
            nn.BatchNorm2d(o),
            nn.ReLU(True)
        )
    def forward(self, x):
        return self.b(x)

def make_enc():
    return nn.Sequential(
        CB(3,32),
        CB(32,64,s=2), CB(64,64),
        CB(64,128,s=2), CB(128,128),
        CB(128,256,s=2), CB(256,256),
        CB(256,512,s=2)
    )

def make_dec():
    return nn.Sequential(
        DB(512,256), CB(256,256),
        DB(256,128), CB(128,128),
        DB(128,64), CB(64,64),
        DB(64,32),
        nn.Conv2d(32,3,3,1,1),
        nn.Sigmoid()
    )

class CNNAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = make_enc()
        self.decoder = make_dec()

    def forward(self,x):
        z = self.encoder(x)
        return self.decoder(z), z

class ResNet18Encoder(nn.Module):
    def __init__(self, freeze_stem=True):
        super().__init__()
        bb = tvm.wide_resnet50_2(weights=tvm.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
        self.stem = nn.Sequential(bb.conv1,bb.bn1,bb.relu,bb.maxpool)
        self.layer1 = bb.layer1
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3
        self.layer4 = bb.layer4

        if freeze_stem:
            for m in [self.stem, self.layer1]:
                for p in m.parameters():
                    p.requires_grad = False

    def forward(self,x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.layer4(x)

class ResNetAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = ResNet18Encoder()
        # WideResNet50-2 layer4 emits 2048 channels. The previous
        # decoder started at 512 channels, causing a ConvTranspose2d mismatch.
        # Keep the five upsampling stages and the original reconstruction head.
        self.dec = nn.Sequential(
            DB(2048,512), CB(512,512),
            DB(512,256), CB(256,256),
            DB(256,128), CB(128,128),
            DB(128,64), CB(64,64),
            DB(64,32),
            nn.Conv2d(32,3,3,1,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        z = self.enc(x)
        return self.dec(z), z

class MemModule(nn.Module):
    def __init__(self, mem_dim, fea_dim, shrink_t=MEM_SHRINK):
        super().__init__()
        self.mem = nn.Parameter(torch.randn(mem_dim, fea_dim))
        nn.init.xavier_normal_(self.mem)
        self.t = shrink_t

    def forward(self,q):
        B,C,H,W = q.shape
        qf = q.permute(0,2,3,1).reshape(-1,C)
        qn = F.normalize(qf,2,1)
        mn = F.normalize(self.mem,2,1)
        at = F.softmax(qn @ mn.t(),1)
        at = F.relu(at-self.t) + self.t*(at>=self.t).float()
        at = at/(at.sum(1,keepdim=True)+1e-8)
        out = (at @ self.mem).reshape(B,H,W,C).permute(0,3,1,2)
        return out, at.reshape(B,H*W,-1)

# MemAE baseline configuration -- required before MemAE() is instantiated
MEM_DIM = 100

class MemAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = make_enc()
        self.mem = MemModule(MEM_DIM,512)
        self.decoder = make_dec()

    def forward(self,x):
        z = self.encoder(x)
        r,at = self.mem(z)
        return self.decoder(r),z,at


# ============================================================
# AMEM-NET V3
# ============================================================

class MultiScaleWideResNet50(nn.Module):
    """
    Frozen ImageNet WideResNet50-2.
    Layer2 (512 channels) and upsampled Layer3 (1024 channels) are concatenated
    to produce 1536-D dense patch features at the Layer2 spatial resolution.
    """
    def __init__(self):
        super().__init__()
        bb = tvm.wide_resnet50_2(weights=tvm.Wide_ResNet50_2_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(
            bb.conv1, bb.bn1, bb.relu, bb.maxpool
        )
        self.layer1 = bb.layer1
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3

        for p in self.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self,x):
        x = self.stem(x)
        x = self.layer1(x)
        f2 = self.layer2(x)       # 512 channels
        f3 = self.layer3(f2)      # 1024 channels
        f3 = F.interpolate(
            f3, size=f2.shape[-2:],
            mode='bilinear', align_corners=False
        )
        return torch.cat([f2,f3],dim=1)  # 1536 channels


class AMEMRouter(nn.Module):
    """
    Dynamic image-level expert router.

    Input:
        global mean-pooled WideResNet50-2 multi-scale feature

    Output:
        expert probabilities
    """
    def __init__(self, dim, n_exp):
        super().__init__()
        h = max(64, dim//2)
        self.net = nn.Sequential(
            nn.Linear(dim,h),
            nn.GELU(),
            nn.LayerNorm(h),
            nn.Linear(h,n_exp)
        )

    def forward(self,g):
        return F.softmax(self.net(g),dim=1)


class AMEMNet(nn.Module):
    """
    AMEM-Net V6.

    The normal memory is populated before router training.
    The router is the adaptive component.

    Expert reconstruction error:
        distance of each query patch to its nearest normal patch
        in each expert memory.

    Final score:
        router-weighted expert error combined with reconstruction-error
        feedback. This keeps the approved multi-expert + dynamic-routing
        + reconstruction-feedback structure without the V2 bottleneck.
    """
    def __init__(self,n_exp=N_EXPERTS,mem_slots=EXPERT_MEMORY_SLOTS):
        super().__init__()
        self.backbone = MultiScaleWideResNet50()
        self.dim = 1536
        self.router = AMEMRouter(self.dim,n_exp)
        self.n_exp = n_exp
        self.mem_slots = mem_slots

        self.register_buffer(
            'expert_memories',
            torch.zeros(n_exp,mem_slots,self.dim)
        )
        self.register_buffer(
            'expert_centroids',
            torch.zeros(n_exp,self.dim)
        )
        self.register_buffer(
            'memory_ready',
            torch.tensor(False)
        )

    @torch.no_grad()
    def encode(self,x):
        return self.backbone(x)

    def global_feature(self,f):
        return F.normalize(f.mean((2,3)),dim=1)

    def _expert_distances(self,f, max_slots=None):
        B,C,H,W = f.shape
        p = F.normalize(
            f.permute(0,2,3,1).reshape(B,H*W,C),
            dim=-1
        )

        mem = F.normalize(self.expert_memories,dim=-1)

        if max_slots is not None:
            max_slots = min(int(max_slots), mem.shape[1])
            mem = mem[:, :max_slots]

        # Compute one expert at a time to keep memory use safe on 4GB GPUs.
        dlist = []
        for e in range(self.n_exp):
            sim = torch.einsum(
                'bpc,mc->bpm',
                p, mem[e]
            )
            best = sim.max(dim=-1).values
            dlist.append(1.0-best)

        return torch.stack(dlist,dim=1)  # B,E,P

    @staticmethod
    def combine_expert_weights(router_w, feedback_w):
        """Shared train/inference weighting rule; prevents objective mismatch."""
        return F.softmax(
            torch.log(router_w + 1e-8) +
            0.50 * torch.log(feedback_w + 1e-8),
            dim=1
        )

    def score_features(self,f):
        B,C,H,W = f.shape
        g = self.global_feature(f)
        router_w = self.router(g)

        d = self._expert_distances(f)
        expert_mean = d.mean(-1)

        # Reconstruction-error feedback:
        # experts with larger reconstruction/memory error receive more
        # influence on anomalous regions.  This avoids the V3 failure mode
        # where negative-error feedback reinforced the easiest (normal)
        # expert and suppressed recall.
        feedback_w = F.softmax(
            expert_mean / FEEDBACK_TEMP,
            dim=1
        )

        # Dynamic routing remains the primary specialization signal;
        # reconstruction-error feedback adaptively reweights it.
        combined_w = F.softmax(
            torch.log(router_w + 1e-8) +
            0.50 * torch.log(feedback_w + 1e-8),
            dim=1
        )

        routed_map = (d * combined_w.unsqueeze(-1)).sum(1)

        # A patch is anomalous only when even the best normal expert cannot
        # explain it.  This is the conservative multi-expert memory signal.
        best_expert_map = d.min(dim=1).values

        # Balanced fusion: preserve the strongest normality evidence while
        # allowing adaptive routing/error feedback to contribute.
        final_map = 0.55*routed_map + 0.45*best_expert_map

        k = max(1,int(final_map.shape[1]*TOP_PATCH_FRAC))
        image_score = final_map.topk(k,dim=1).values.mean(1)

        return image_score, final_map, d, router_w, feedback_w

    def forward(self,x):
        f = self.encode(x)
        return self.score_features(f)


# ============================================================
# NORMAL MEMORY CONSTRUCTION
# ============================================================

@torch.no_grad()
def extract_normal_bank(model, loader):
    """
    Extract:
      patch_bank: all normal patch features
      global_bank: one normalized global feature per image

    The global bank is used only to discover normal modes for the expert
    memories. No test images are involved.
    """
    model.eval()
    patch_bank = []
    global_bank = []

    for imgs,_ in tqdm(loader,leave=False,desc='  extracting normal bank'):
        imgs = imgs.to(DEVICE)
        f = model.encode(imgs)
        B,C,H,W = f.shape

        patch_bank.append(
            f.permute(0,2,3,1).reshape(B*H*W,C).cpu().numpy()
        )
        global_bank.append(
            F.normalize(f.mean((2,3)),dim=1).cpu().numpy()
        )

    return (
        np.vstack(patch_bank).astype(np.float32),
        np.vstack(global_bank).astype(np.float32)
    )


def build_expert_memories(model, loader):
    """
    Create expert memories from normal training images.

    Expert specialization is image-level:
        global normal feature → 4 normal modes

    Patches belonging to each normal mode are stored in that expert.
    This is substantially less destructive than V2's direct patch-space
    K-Means partitioning.
    """
    # The training loader shuffles each iteration. Use a deterministic, non-shuffled
    # view for BOTH feature extraction passes so image-mode labels stay aligned
    # with each image's patch features.
    ordered_loader = DataLoader(
        loader.dataset,
        batch_size=loader.batch_size or BATCH_SIZE,
        shuffle=False,
        num_workers=loader.num_workers,
        pin_memory=(DEVICE.type == 'cuda')
    )
    patch_bank, global_bank = extract_normal_bank(model, ordered_loader)

    if len(patch_bank) > MAX_BANK_PATCHES:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(
            len(patch_bank),
            MAX_BANK_PATCHES,
            replace=False
        )
        patch_bank = patch_bank[idx]

    # Discover normal image modes.
    km = MiniBatchKMeans(
        n_clusters=N_EXPERTS,
        random_state=SEED,
        batch_size=max(1, BATCH_SIZE),
        n_init=10,
        max_iter=200
    )
    image_labels = km.fit_predict(global_bank)

    # We need patch-to-image assignment.
    # Re-extract image-wise patches so every image's patches go to its
    # corresponding expert. This is intentionally explicit and deterministic.
    expert_patches = [[] for _ in range(N_EXPERTS)]

    model.eval()
    with torch.no_grad():
        img_index = 0
        for imgs,_ in tqdm(ordered_loader,leave=False,desc='  assigning patches to experts'):
            imgs = imgs.to(DEVICE)
            f = model.encode(imgs)
            B,C,H,W = f.shape
            pf = f.permute(0,2,3,1).reshape(B,H*W,C).cpu().numpy()

            for j in range(B):
                e = int(image_labels[img_index+j])
                expert_patches[e].append(
                    pf[j].astype(np.float32)
                )
            img_index += B

    rng = np.random.default_rng(SEED)

    for e in range(N_EXPERTS):
        if len(expert_patches[e]) == 0:
            # Safety fallback.
            xe = patch_bank
        else:
            xe = np.concatenate(expert_patches[e],axis=0)

        # Sample a large dense normal memory.
        if len(xe) > EXPERT_MEMORY_SLOTS:
            idx = rng.choice(
                len(xe),
                EXPERT_MEMORY_SLOTS,
                replace=False
            )
            xe = xe[idx]
        elif len(xe) < EXPERT_MEMORY_SLOTS:
            idx = rng.choice(
                len(xe),
                EXPERT_MEMORY_SLOTS,
                replace=True
            )
            xe = xe[idx]

        xe = sk_normalize(xe.astype(np.float32),axis=1)
        model.expert_memories[e].copy_(
            torch.from_numpy(xe).to(DEVICE)
        )
        model.expert_centroids[e].copy_(
            torch.from_numpy(xe.mean(0)).to(DEVICE)
        )

    model.memory_ready.fill_(True)

    print(
        f'  V6 memory ready: {N_EXPERTS} experts × '
        f'{EXPERT_MEMORY_SLOTS} dense normal patches'
    )


# ============================================================
# SELF-SUPERVISED PSEUDO ANOMALIES
# ============================================================

def make_pseudo_anomaly_batch(imgs):
    """
    Create local pseudo defects from normal images.

    Three corruption families:
      1. Cut-and-paste with intensity shift
      2. Gaussian-noise rectangle
      3. Strong local contrast shift

    Returns:
      pseudo images
      binary input-space masks

    These masks are synthetic labels only and never come from MVTec test labels.
    """
    B,C,H,W = imgs.shape
    out = imgs.clone()
    masks = torch.zeros(
        B,1,H,W,
        device=imgs.device
    )

    for i in range(B):
        ph = random.randint(max(6,H//12),max(8,H//5))
        pw = random.randint(max(6,W//12),max(8,W//5))
        ph = min(ph,H)
        pw = min(pw,W)

        y = random.randint(0,max(0,H-ph))
        x = random.randint(0,max(0,W-pw))

        mode = random.randint(0,2)

        if mode == 0:
            sy = random.randint(0,max(0,H-ph))
            sx = random.randint(0,max(0,W-pw))
            patch = imgs[i,:,sy:sy+ph,sx:sx+pw].clone()
            patch = (patch*random.uniform(0.45,1.55)).clamp(-3,3)
            out[i,:,y:y+ph,x:x+pw] = patch

        elif mode == 1:
            patch = imgs[i,:,y:y+ph,x:x+pw].clone()
            noise = torch.randn_like(patch)*random.uniform(0.5,1.2)
            out[i,:,y:y+ph,x:x+pw] = (patch+noise).clamp(-3,3)

        else:
            patch = imgs[i,:,y:y+ph,x:x+pw].clone()
            mean = patch.mean()
            out[i,:,y:y+ph,x:x+pw] = (
                (patch-mean)*random.uniform(1.8,3.0)+mean
            ).clamp(-3,3)

        masks[i,0,y:y+ph,x:x+pw] = 1.0

    return out,masks


def patch_mask(mask,h,w):
    return F.interpolate(
        mask,
        size=(h,w),
        mode='nearest'
    ).flatten(1)


# ============================================================
# V6 ROUTER TRAINING
# ============================================================

def train_amem_router(model, trl, epochs=EPOCHS, cat='unknown'):
    """Train router with an atomic full-state checkpoint after every epoch."""
    model.train()
    for p in model.backbone.parameters():
        p.requires_grad = False

    opt = optim.AdamW(model.router.parameters(), lr=ROUTER_LR,
                      weight_decay=ROUTER_WEIGHT_DECAY)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=ROUTER_LR/20)
    ckpt_path = _checkpoint_path('amem_router', cat)
    start_epoch = 1
    # Persist actual per-epoch router training losses so resumed runs keep one continuous curve.
    loss_history = []

    if os.path.isfile(ckpt_path):
        ckpt = _load_checkpoint(ckpt_path)
        if ckpt.get('router_state') is not None:
            model.router.load_state_dict(ckpt['router_state'])
        elif ckpt.get('model_state') is not None:  # compatibility with an earlier full-state checkpoint
            model.load_state_dict(ckpt['model_state'])
        if ckpt.get('optimizer_state') is not None:
            opt.load_state_dict(ckpt['optimizer_state'])
        if ckpt.get('scheduler_state') is not None:
            sch.load_state_dict(ckpt['scheduler_state'])
        _restore_rng_state(ckpt.get('rng_state'))
        loss_history = list(ckpt.get('loss_history', []))
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        print(f'    Resuming AMEM router for {cat} at epoch {start_epoch}/{epochs}')

    if start_epoch > epochs:
        print(f'    AMEM router for {cat} already completed ({epochs} epochs); skipping training.')
        return

    for ep in range(start_epoch, epochs+1):
        model.train()
        running = 0.0
        for imgs,_ in trl:
            imgs = imgs.to(DEVICE)
            with torch.no_grad():
                f = model.encode(imgs)
                d = model._expert_distances(f, max_slots=ROUTER_MEMORY_PER_EXPERT)
                expert_err = d.mean(-1)
                target_w = F.softmax(-expert_err/FEEDBACK_TEMP, dim=1)
                pseudo,_ = make_pseudo_anomaly_batch(imgs)
                pf = model.encode(pseudo)
                pd = model._expert_distances(pf, max_slots=ROUTER_MEMORY_PER_EXPERT)
                pseudo_err = pd.mean(-1)

            g = model.global_feature(f)
            router_w = model.router(g)
            kl = F.kl_div(torch.log(router_w+1e-8), target_w.detach(), reduction='batchmean')
            normal_feedback = F.softmax(expert_err/FEEDBACK_TEMP, dim=1)
            pseudo_feedback = F.softmax(pseudo_err/FEEDBACK_TEMP, dim=1)
            normal_combined = model.combine_expert_weights(router_w, normal_feedback)
            pseudo_combined = model.combine_expert_weights(router_w, pseudo_feedback)
            routed_normal = (expert_err * normal_combined).sum(1)
            routed_pseudo = (pseudo_err * pseudo_combined).sum(1)
            margin_loss = F.relu(PSEUDO_MARGIN + routed_normal - routed_pseudo).mean()
            usage = router_w.mean(0)
            entropy = -(usage*torch.log(usage+1e-8)).sum()
            balance_penalty = F.relu(math.log(N_EXPERTS)*0.60 - entropy)
            loss = ROUTER_KL_WEIGHT*kl + 0.50*margin_loss + 0.01*balance_penalty
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.router.parameters(), 1.0)
            opt.step()
            running += float(loss.item())

        sch.step()
        epoch_loss = running / max(1, len(trl))
        loss_history.append({'epoch': int(ep), 'train_loss': float(epoch_loss)})
        # Save model + optimizer + scheduler + RNG + loss history after EVERY completed epoch.
        _atomic_torch_save({
            'model_type': 'amem_router', 'category': cat,
            'epoch': ep, 'total_epochs': epochs,
            'router_state': model.router.state_dict(),
            'optimizer_state': opt.state_dict(),
            'scheduler_state': sch.state_dict(),
            'rng_state': _rng_checkpoint_state(),
            'loss_history': loss_history,
            'config': {'n_experts': N_EXPERTS, 'memory_slots': EXPERT_MEMORY_SLOTS,
                       'img_size': IMG_SIZE, 'seed': SEED}
        }, ckpt_path)
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f'    AMEM V6 epoch {ep:02d}/{epochs} router-loss={epoch_loss:.5f} | checkpoint saved')


# ============================================================
# SCORE COLLECTION / THRESHOLD
# ============================================================

@torch.no_grad()
def collect_amem_scores(model, loader):
    model.eval()
    scores = []
    labels = []
    maps = []
    origs = []

    for imgs,lbls in loader:
        imgs = imgs.to(DEVICE)
        sc, amap, _, _, _ = model(imgs)

        scores.extend(sc.cpu().numpy())
        labels.extend(lbls.numpy())
        maps.extend(amap.cpu().numpy())
        origs.extend(list(imgs.cpu().unbind(0)))

    return (
        np.asarray(scores),
        np.asarray(labels),
        maps,
        origs
    )


@torch.no_grad()
def collect_pseudo_validation_scores(model, loader):
    """
    Generate multiple pseudo-anomaly variants from held-out normal validation
    images. Used only for threshold calibration.
    """
    model.eval()
    normal_scores = []
    pseudo_scores = []

    for imgs,_ in loader:
        imgs = imgs.to(DEVICE)

        sc,_,_,_,_ = model(imgs)
        normal_scores.extend(sc.cpu().numpy())

        for _ in range(4):
            pimgs,_ = make_pseudo_anomaly_batch(imgs)
            psc,_,_,_,_ = model(pimgs)
            pseudo_scores.extend(psc.cpu().numpy())

    return (
        np.asarray(normal_scores),
        np.asarray(pseudo_scores)
    )


def choose_validation_threshold(normal_scores, pseudo_scores):
    """Choose a threshold by maximizing Youden's J on validation-only data.

    The positive validation examples are synthetic pseudo-anomalies generated
    from held-out normal training images; real MVTec test labels are never used.
    This is a threshold-calibration procedure, not a standalone novelty claim.
    """
    normal_scores = np.asarray(normal_scores, dtype=np.float64)
    pseudo_scores = np.asarray(pseudo_scores, dtype=np.float64)
    normal_scores = normal_scores[np.isfinite(normal_scores)]
    pseudo_scores = pseudo_scores[np.isfinite(pseudo_scores)]
    if len(normal_scores) == 0 or len(pseudo_scores) == 0:
        raise ValueError('Youden-J calibration requires finite normal and pseudo-anomaly validation scores.')

    all_scores = np.concatenate([normal_scores, pseudo_scores])
    candidates = np.unique(np.quantile(all_scores, np.linspace(0.01, 0.99, 500)))
    best_threshold = float(candidates[0])
    best_j = -np.inf
    for threshold in candidates:
        tpr = float((pseudo_scores >= threshold).mean())
        fpr_internal = float((normal_scores >= threshold).mean())
        j = tpr - fpr_internal
        if j > best_j:
            best_j = j
            best_threshold = float(threshold)
    return best_threshold, float(best_j)


# ============================================================
# AMEM TRAIN + EVALUATE
# ============================================================

def train_and_eval_amem(cat):
    trl,vll,tel,n1 = get_loaders(cat)

    if n1 == 0:
        return None

    model = AMEMNet().to(DEVICE)

    amem_ckpt = _checkpoint_path('amem_router', cat)
    memory_path = os.path.join(MODEL_DIR, f'amem_v6_{cat}_memory.pt')
    if os.path.isfile(memory_path):
        mem_state = _load_checkpoint(memory_path)
        model.expert_memories.copy_(mem_state['expert_memories'].to(DEVICE))
        model.expert_centroids.copy_(mem_state['expert_centroids'].to(DEVICE))
        model.memory_ready.copy_(mem_state['memory_ready'].to(DEVICE))
        print('  [1/4] Restored dense expert memories from Drive; skipping memory rebuild.')
    else:
        print('  [1/4] Building dense normal expert memories...')
        build_expert_memories(model,trl)
        # Save the large, frozen memory bank once. Per-epoch router checkpoints stay small.
        _atomic_torch_save({
            'expert_memories': model.expert_memories.detach().cpu(),
            'expert_centroids': model.expert_centroids.detach().cpu(),
            'memory_ready': model.memory_ready.detach().cpu(),
            'category': cat, 'n_experts': N_EXPERTS,
            'memory_slots': EXPERT_MEMORY_SLOTS, 'img_size': IMG_SIZE,
            'seed': SEED
        }, memory_path)
        print('  Saved frozen expert memory bank:', memory_path)

    print('  [2/4] Training dynamic router with self-supervised margin...')
    train_amem_router(model,trl,EPOCHS,cat=cat)

    # Save final model weights as a convenient inference artifact.
    _atomic_torch_save(model.state_dict(), f'{MODEL_DIR}/amem_v6_{cat}.pth')

    print('  [3/4] Validation-only threshold calibration...')
    normal_val,pseudo_val = collect_pseudo_validation_scores(
        model,vll
    )
    threshold,gap = choose_validation_threshold(
        normal_val,pseudo_val
    )

    print(
        f'    validation pseudo gap={gap:.4f}, '
        f'threshold={threshold:.6f}'
    )

    print('  [4/4] Test evaluation (labels used ONLY after threshold is frozen)...')
    sc,lb,maps,origs = collect_amem_scores(model,tel)

    mt = metrics_at_threshold(
        sc,lb,threshold
    )

    return (
        model,tel,vll,
        mt,sc,lb,maps,origs,
        threshold
    )


# ============================================================
# BASELINE TRAINING HELPERS
# ============================================================

def score_components(pred,tgt,z,mem_z):
    mse = F.mse_loss(
        pred,tgt,reduction='none'
    ).mean(1)
    flat = mse.flatten(1)
    k = max(1,int(flat.shape[1]*0.10))
    rec = flat.topk(k,1).values.mean(1)
    return rec


@torch.no_grad()
def collect_scores(model,mtype,loader):
    model.eval()
    recs=[]

    for imgs,_ in loader:
        imgs=imgs.to(DEVICE)
        tgt=to_01(imgs)

        if mtype=='memae':
            r,z,at=model(imgs)
        else:
            r,z=model(imgs)

        recs.extend(
            score_components(r,tgt,z,z).cpu().numpy()
        )

    return np.asarray(recs)


@torch.no_grad()
def collect_baseline_maps(model,mtype,loader):
    model.eval(); maps=[]
    for imgs,_ in loader:
        imgs=imgs.to(DEVICE); tgt=to_01(imgs)
        if mtype=='memae': r,z,_=model(imgs)
        else: r,z=model(imgs)
        amap=F.mse_loss(r,tgt,reduction='none').mean(1)
        maps.extend(list(amap.detach().cpu().numpy()))
    return maps


@torch.no_grad()
def eval_model(model,mtype,tel):
    model.eval()

    scores=[]
    labels=[]
    origs=[]
    recons=[]
    maps=[]

    for imgs,lbls in tel:
        imgs=imgs.to(DEVICE)
        tgt=to_01(imgs)

        if mtype=='memae':
            r,z,_=model(imgs)
        else:
            r,z=model(imgs)

        sc=score_components(r,tgt,z,z)

        scores.extend(sc.cpu().numpy())
        labels.extend(lbls.numpy())

        origs.extend(list(imgs.cpu().unbind(0)))
        recons.extend(list(r.cpu().unbind(0)))

        maps.extend(
            list(
                F.mse_loss(
                    r,tgt,reduction='none'
                ).mean(1,keepdim=True).cpu().unbind(0)
            )
        )

    return (
        np.asarray(scores),
        np.asarray(labels),
        origs,recons,maps
    )


def train_model(model,mtype,cat,epochs=EPOCHS):
    best_path = f'{MODEL_DIR}/{mtype}_{cat}.pth'
    ckpt_path = _checkpoint_path(mtype, cat)
    trl,vll,tel,n1 = get_loaders(cat)
    if n1 == 0:
        return None,None,None

    model = model.to(DEVICE)
    opt = optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()), lr=LR, weight_decay=2e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, epochs, LR/20)
    scaler = GradScaler(enabled=DEVICE.type=='cuda')
    best = float('inf')
    start_epoch = 1

    if os.path.isfile(ckpt_path):
        ckpt = _load_checkpoint(ckpt_path)
        model.load_state_dict(ckpt['model_state'])
        if ckpt.get('optimizer_state') is not None: opt.load_state_dict(ckpt['optimizer_state'])
        if ckpt.get('scheduler_state') is not None: sch.load_state_dict(ckpt['scheduler_state'])
        if ckpt.get('scaler_state') is not None: scaler.load_state_dict(ckpt['scaler_state'])
        best = float(ckpt.get('best_val', best))
        _restore_rng_state(ckpt.get('rng_state'))
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        print(f'    Resuming {mtype}/{cat} at epoch {start_epoch}/{epochs}')

    if start_epoch <= epochs:
        for ep in range(start_epoch, epochs+1):
            model.train()
            total = 0.0
            for imgs,_ in trl:
                imgs = imgs.to(DEVICE)
                tgt = to_01(imgs)
                opt.zero_grad(set_to_none=True)
                with autocast(enabled=DEVICE.type=='cuda'):
                    if mtype == 'memae':
                        r,z,at = model(imgs)
                        loss = F.smooth_l1_loss(r,tgt) + ENTROPY_W*entropy_loss(at)
                    else:
                        r,z = model(imgs)
                        loss = F.smooth_l1_loss(r,tgt)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt)
                scaler.update()
                total += float(loss.item())

            sch.step()
            model.eval()
            v = 0.0
            with torch.no_grad():
                for imgs,_ in vll:
                    imgs = imgs.to(DEVICE)
                    tgt = to_01(imgs)
                    r = model(imgs)[0]
                    v += F.smooth_l1_loss(r,tgt).item()
            v /= max(1,len(vll))
            if v < best:
                best = v
                _atomic_torch_save(model.state_dict(), best_path)

            # Full restart checkpoint after each completed epoch.
            _atomic_torch_save({
                'model_type': mtype, 'category': cat,
                'epoch': ep, 'total_epochs': epochs,
                'model_state': model.state_dict(),
                'optimizer_state': opt.state_dict(),
                'scheduler_state': sch.state_dict(),
                'scaler_state': scaler.state_dict(),
                'best_val': best,
                'rng_state': _rng_checkpoint_state(),
                'config': {'img_size': IMG_SIZE, 'seed': SEED}
            }, ckpt_path)
            if ep == 1 or ep % 5 == 0 or ep == epochs:
                print(f'    {mtype} ep {ep:02d}/{epochs} val={v:.5f} best={best:.5f} | checkpoint saved')
    else:
        print(f'    {mtype}/{cat} already reached epoch {epochs}; loading best weights for evaluation.')

    if os.path.isfile(best_path):
        model.load_state_dict(_load_checkpoint(best_path))
    return model,tel,vll


# ============================================================
# RUN ALL MODELS
# ============================================================

MODEL_CFGS=[
    ('cnnae',CNNAutoencoder,'CNN-AE'),
    ('resnetae_wrn50',ResNetAE,'ResNet-AE'),
    ('memae',MemAE,'MemAE')
]

CSV_ALL=f'{SAVE_DIR}/all_model_results.csv'

if os.path.isfile(CSV_ALL):
    try:
        _loaded_results = pd.read_csv(CSV_ALL)
        if 'Category' in _loaded_results.columns:
            _loaded_results = _loaded_results[_loaded_results['Category'].astype(str).isin(ALL_CATS)]
        all_results = _loaded_results.to_dict('records')
        print(f'Restored {len(all_results)} saved result rows from Drive.')
    except Exception as exc:
        print('Could not read prior results CSV; starting a fresh in-memory table:', exc)
        all_results = []
else:
    all_results = []
ae_roc={}
ae_viz={}

for cat in ALL_CATS:

    if not os.path.exists(os.path.join(DATASET_PATH,cat)):
        continue

    print(f'\n{"="*70}\n{cat.upper()}\n{"="*70}')
    ae_roc[cat]={}

    # ---------- three neural baselines ----------
    for mtype,ModelClass,label in MODEL_CFGS:

        torch.cuda.empty_cache()

        model=ModelClass()

        result=train_model(
            model,mtype,cat,EPOCHS
        )

        if result[0] is None:
            continue

        model,tel,vll=result

        vsc=collect_scores(
            model,mtype,vll
        )
        val_pixel_maps=collect_baseline_maps(model,mtype,vll)

        threshold=normal_quantile_threshold(
            vsc,THRESH_Q
        )

        sc,lb,origs,recons,rmaps=eval_model(
            model,mtype,tel
        )
        pix=pixel_metrics_for_category(cat, [p for p,_ in tel.dataset.data], rmaps, val_pixel_maps)

        mt=metrics_at_threshold(
            sc,lb,threshold
        )

        ae_roc[cat][label]={
            'fpr':mt['fpr_c'],
            'tpr':mt['tpr_c'],
            'auroc':mt['AUROC']
        }

        _new_row = {
            'Category':cat,
            'Model':label,
            'AUROC':mt['AUROC'],
            'AUC-PR':mt['AUCPR'],
            'Accuracy':mt['Accuracy'],
            'Precision':mt['Precision'],
            'Recall':mt['Recall'],
            'F1':mt['F1'],
            **{k:v for k,v in pix.items() if k != 'Pixel-IoU'}
        }
        _upsert_result(all_results, _new_row)
        _csv_tmp = CSV_ALL + '.tmp'
        pd.DataFrame(all_results).drop(columns=['FPR','Pixel-IoU','PixelThreshold'], errors='ignore').to_csv(_csv_tmp, index=False)
        os.replace(_csv_tmp, CSV_ALL)
        print(f'Saved results immediately: {cat} / {label}')

        print(
            f"  {label:<10} "
            f"AUROC={mt['AUROC']:5.2f}% "
            f"F1={mt['F1']:5.2f}% "
            f"Prec={mt['Precision']:5.2f}% "
            f"Rec={mt['Recall']:5.2f}% "
            f"Acc={mt['Accuracy']:5.2f}% "
        )

        del model
        torch.cuda.empty_cache()


    # ---------- proposed AMEM-Net V6 ----------
    torch.cuda.empty_cache()

    print('\n  >>> AMEM-Net V6')

    result=train_and_eval_amem(cat)

    if result is not None:

        (
            model,tel,vll,
            mt,sc,lb,maps,origs,
            threshold
        )=result
        # Pixel IoU calibration uses only held-out normal validation anomaly maps.
        _,_,val_maps_normal,_ = collect_amem_scores(model,vll)
        pix = pixel_metrics_for_category(
            cat, [p for p,_ in tel.dataset.data], maps, val_maps_normal
        )

        ae_roc[cat]['AMEM-Net']={
            'fpr':mt['fpr_c'],
            'tpr':mt['tpr_c'],
            'auroc':mt['AUROC']
        }

        ae_viz[cat]={
            'origs':origs,
            'maps':[
                torch.tensor(m).unsqueeze(0)
                for m in maps
            ],
            'labels':lb.tolist(),
            'scores':sc.tolist(),
            'threshold':threshold
        }

        _new_row = {
            'Category':cat,
            'Model':'AMEM-Net',
            'AUROC':mt['AUROC'],
            'AUC-PR':mt['AUCPR'],
            'Accuracy':mt['Accuracy'],
            'Precision':mt['Precision'],
            'Recall':mt['Recall'],
            'F1':mt['F1'],
            **{k:v for k,v in pix.items() if k != 'Pixel-IoU'}
        }
        _upsert_result(all_results, _new_row)
        _csv_tmp = CSV_ALL + '.tmp'
        pd.DataFrame(all_results).drop(columns=['FPR','Pixel-IoU','PixelThreshold'], errors='ignore').to_csv(_csv_tmp, index=False)
        os.replace(_csv_tmp, CSV_ALL)
        print(f'Saved results immediately: {cat} / AMEM-Net')

        print(
            f"  AMEM-Net  "
            f"AUROC={mt['AUROC']:5.2f}% "
            f"F1={mt['F1']:5.2f}% "
            f"Prec={mt['Precision']:5.2f}% "
            f"Rec={mt['Recall']:5.2f}% "
            f"Acc={mt['Accuracy']:5.2f}% "
        )

        del model
        torch.cuda.empty_cache()

    pd.DataFrame(all_results).drop(columns=['FPR','Pixel-IoU','PixelThreshold'], errors='ignore').to_csv(
        CSV_ALL,index=False
    )


# Build and validate the final table without assuming columns exist.
df_all = pd.DataFrame(all_results)
if not df_all.empty and 'Category' in df_all.columns:
    df_all = df_all[df_all['Category'].astype(str).isin(ALL_CATS)].copy()
    all_results = df_all.to_dict('records')
    df_all.to_csv(CSV_ALL, index=False)

print('\n' + '='*80)
print('FINAL MEANS — MEASURED RESULTS ONLY')
print('='*80)
print('Result records:', len(df_all))
print('Result columns:', df_all.columns.tolist())

if df_all.empty:
    print('NO RESULTS RECORDED. Check the dataset path and whether evaluation completed.')
else:
    # Accept common capitalization/alias variations without inventing data.
    rename = {}
    for col in df_all.columns:
        key = str(col).strip().lower().replace('_', ' ').replace('-', ' ')
        if key in ('model', 'model name', 'method', 'detector'):
            rename[col] = 'Model'
        elif key in ('category', 'class', 'object category'):
            rename[col] = 'Category'
        elif key in ('aucpr', 'auc pr', 'aupr'):
            rename[col] = 'AUC-PR'
    df_all = df_all.rename(columns=rename)

    required_metrics = [
        'AUROC', 'AUC-PR', 'Accuracy', 'Precision', 'Recall', 'F1',
        'Pixel-AUROC', 'Pixel-AUPR', 'AUPRO'
    ]
    missing_metrics = [c for c in required_metrics if c not in df_all.columns]

    if 'Model' not in df_all.columns:
        print("ERROR: The result records have no model-name field; cannot group by model.")
        print('First record for diagnosis:', all_results[0] if all_results else None)
        print('No model averages were generated.')
    elif missing_metrics:
        print('ERROR: Missing metric columns:', missing_metrics)
        print('Available columns:', df_all.columns.tolist())
        print('No incomplete metric table was reported as final.')
    else:
        # Only rows actually recorded by completed evaluations are summarized.
        for col in required_metrics:
            df_all[col] = pd.to_numeric(df_all[col], errors='coerce')
        df_all.to_csv(CSV_ALL, index=False)
        print('\nMean metrics by model (measured records only):')
        print(df_all.groupby('Model')[required_metrics].mean().round(2).to_string())
        if 'Category' in df_all.columns:
            print('\nMetrics by category and model:')
            print(df_all.groupby(['Category', 'Model'])[required_metrics].mean().round(2).to_string())
        print('\nSaved measured results to:', CSV_ALL)

print('\nThresholds are calibrated using validation-normal data, not test labels.')
print('AMEM-Net V6 evaluation cell finished; verify the record count and coverage before reporting results.')

In [20]:
# CELL 6 — Load  measured results and show the full results table
from pathlib import Path
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPORT_DIR = Path("AMEM_Net_IEEE_Access_Actual_Results")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer the exact output paths defined by the project cell.
_result_candidates = []
if "CSV_ALL" in globals():
    _result_candidates.append(Path(CSV_ALL))
if "SAVE_DIR" in globals():
    _result_candidates.extend([Path(SAVE_DIR) / "all_model_results.csv",
                               Path(SAVE_DIR) / "patchcore_results.csv"])
_result_candidates.extend([
    Path("AMEM_Net_V6_Laptop/results_v6_wrn50_youden_8cats/all_model_results.csv"),
    Path("AMEM_Net_V6_Laptop/results_v6_wrn50_youden_8cats/patchcore_results.csv"),
])

def _read_first_existing(candidates):
    seen = set()
    for p in candidates:
        try:
            p = Path(p)
            if str(p) in seen:
                continue
            seen.add(str(p))
            if p.is_file() and p.stat().st_size > 0:
                return pd.read_csv(p), p
        except Exception:
            pass
    return pd.DataFrame(), None

df_actual, actual_csv_path = _read_first_existing(_result_candidates)
if df_actual.empty:
    raise FileNotFoundError(
        "No saved measured result CSV was found. Run the original project evaluation cell first. "
        "Expected all_model_results.csv in SAVE_DIR."
    )

# Normalize common column-name variants
rename = {}
for col in df_actual.columns:
    key = str(col).strip().lower().replace("_", " ").replace("-", " ")
    if key in ("category", "class", "object category"): rename[col] = "Category"
    elif key in ("model", "model name", "method", "detector"): rename[col] = "Model"
    elif key in ("aucpr", "auc pr", "aupr"): rename[col] = "AUC-PR"
    elif key in ("pixel auroc",): rename[col] = "Pixel-AUROC"
    elif key in ("pixel aupr",): rename[col] = "Pixel-AUPR"
    elif key in ("aupro",): rename[col] = "AUPRO"
df_actual = df_actual.rename(columns=rename)

for col in ("Category", "Model"):
    if col not in df_actual.columns:
        raise ValueError(f"Saved result CSV is missing required column {col!r}. Columns: {df_actual.columns.tolist()}")

metric_names = ["AUROC", "AUC-PR", "F1", "Precision", "Recall", "Accuracy",
                "Pixel-AUROC", "Pixel-AUPR", "AUPRO"]
for m in metric_names:
    if m in df_actual.columns:
        df_actual[m] = pd.to_numeric(df_actual[m], errors="coerce")

# Keep one measured row per category/model pair 
df_actual = df_actual.drop_duplicates(["Category", "Model"], keep="last").copy()
category_order = ["bottle", "carpet", "hazelnut", "metal_nut", "wood", "leather", "cable", "transistor"]
model_order = ["CNN-AE", "ResNet-AE", "MemAE", "PatchCore", "AMEM-Net"]
df_actual["_cat_key"] = df_actual["Category"].astype(str).str.lower().str.replace(" ", "_", regex=False)
df_actual["_model_order"] = pd.Categorical(df_actual["Model"], categories=model_order, ordered=True)
df_actual["_cat_order"] = pd.Categorical(df_actual["_cat_key"], categories=category_order, ordered=True)
df_actual = df_actual.sort_values(["_cat_order", "_model_order"]).drop(columns=["_cat_key","_cat_order","_model_order"])

display_cols = ["Category", "Model"] + [m for m in metric_names if m in df_actual.columns]
full_results_table = df_actual[display_cols].copy()
print(f"ACTUAL CATEGORY-WISE RESULTS — {len(full_results_table)} recorded category/model rows")
print("Source:", actual_csv_path)
display(full_results_table.style.format({m: "{:.2f}" for m in metric_names if m in full_results_table.columns}).format(na_rep="N/A"))

full_results_table.to_csv(REPORT_DIR / "Actual_Categorywise_Results.csv", index=False)
print("Saved:", REPORT_DIR / "Actual_Categorywise_Results.csv")

available_metrics = [m for m in metric_names if m in df_actual.columns]
macro_means = df_actual.groupby("Model", observed=False)[available_metrics].mean().reindex(
    [m for m in model_order if m in set(df_actual["Model"])]
)
print("\\nMACRO MEANS — recorded values only")
display(macro_means.round(2).style.format("{:.2f}", na_rep="N/A"))
macro_means.to_csv(REPORT_DIR / "Actual_Macro_Mean_Comparison.csv", float_format="%.4f")
print("Saved:", REPORT_DIR / "Actual_Macro_Mean_Comparison.csv")


FULL CATEGORY-WISE RESULTS (40 rows; percentages)


,Category,Model,AUROC,F1,Precision,Recall,Accuracy
0,Bottle,CNN-AE,92.43,86.95,89.45,84.59,83.54
1,Bottle,ResNet-AE,95.61,91.42,93.33,89.59,89.91
2,Bottle,MemAE,95.04,89.60,91.23,88.03,87.96
3,Bottle,PatchCore,99.60,98.44,96.92,100.00,97.59
4,Bottle,AMEM-Net,99.79,98.66,99.02,98.30,98.24
5,Carpet,CNN-AE,88.56,79.29,82.23,76.56,76.63
6,Carpet,ResNet-AE,92.93,89.14,91.23,87.14,86.60
7,Carpet,MemAE,93.18,86.83,89.16,84.62,84.14
8,Carpet,PatchCore,98.52,97.73,98.85,96.63,96.58
9,Carpet,AMEM-Net,99.53,98.19,98.75,97.63,97.42


MACRO-MEAN RESULTS (%)


,AUROC,F1,Precision,Recall,Accuracy
Model,,,,,
CNN-AE,89.43,80.65,84.51,77.16,77.25
ResNet-AE,93.44,88.66,91.61,85.91,85.86
MemAE,92.83,85.75,88.31,83.36,82.73
PatchCore,97.00,92.33,92.36,92.56,90.61
AMEM-Net,97.72,94.84,96.47,93.28,93.37


In [25]:
# CELL 7 — FIGURE 1: AUROC heatmap; FIGURE 2: F1 heatmap 
plt.rcParams.update({"font.family":"serif", "font.size":11, "savefig.dpi":600, "figure.dpi":140,
                     "pdf.fonttype":42, "ps.fonttype":42, "axes.spines.top":False, "axes.spines.right":False})

# Normalize category labels for a stable eight-category ordering.
cat_key = df_actual["Category"].astype(str).str.lower().str.replace(" ", "_", regex=False)
cat_map = dict(zip(cat_key, df_actual["Category"].astype(str)))
cats_present = [c for c in category_order if c in set(cat_key)]
cats_labels = [cat_map[c] for c in cats_present]
models_present = [m for m in model_order if m in set(df_actual["Model"].astype(str))]

def _draw_heatmap(metric, title, filename, vmin):
    if metric not in df_actual.columns or not df_actual[metric].notna().any():
        print(f"{title} not generated: {metric} is absent from the measured results CSV.")
        return
    p = df_actual.pivot_table(index="Category", columns="Model", values=metric, aggfunc="last")
    # Reindex by the actual category labels in the original experiment.
    p.index = p.index.astype(str)
    p = p.reindex(index=cats_labels, columns=models_present)
    mat = p.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(10.5,6.2), constrained_layout=True)
    im = ax.imshow(np.ma.masked_invalid(mat), aspect="auto", vmin=vmin, vmax=100, cmap="viridis")
    ax.set_xticks(np.arange(len(models_present)), labels=models_present, rotation=15, ha="right")
    ax.set_yticks(np.arange(len(cats_labels)), labels=cats_labels)
    ax.set_title(title, pad=14)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i,j]
            ax.text(j,i,"N/A" if not np.isfinite(val) else f"{val:.2f}",ha="center",va="center",fontsize=10,
                    color="white" if (not np.isfinite(val) or val < (vmin+100)/2) else "black")
    fig.colorbar(im, ax=ax, label=f"{metric} (%)", shrink=.85)
    fig.savefig(REPORT_DIR / f"{filename}.png", dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(REPORT_DIR / f"{filename}.pdf", bbox_inches="tight", facecolor="white")
    plt.show(); plt.close(fig)
    print(f"Saved {filename}.png and {filename}.pdf")

_draw_heatmap("AUROC", "Category-wise AUROC (%)", "Fig01_AUROC_Heatmap", 80)
_draw_heatmap("F1", "Category-wise F1-score (%)", "Fig02_F1_Heatmap", 65)


In [26]:
# CELL 8 — FIGURES 3, 4 and 5 
# Figure 3: macro-mean comparison across models for metrics present in the CSV.
metric_names_plot = [m for m in ["AUROC","F1","Precision","Recall","Accuracy"] if m in df_actual.columns and df_actual[m].notna().any()]
models_present = [m for m in model_order if m in set(df_actual["Model"].astype(str))]
if metric_names_plot:
    means_plot = df_actual.groupby("Model")[metric_names_plot].mean().reindex(models_present)
    fig, ax = plt.subplots(figsize=(13,7), constrained_layout=True); x=np.arange(len(metric_names_plot)); width=.15
    for i, model in enumerate(models_present):
        vals=means_plot.loc[model, metric_names_plot].to_numpy(dtype=float)
        bars=ax.bar(x+(i-(len(models_present)-1)/2)*width, vals, width, label=model)
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=7, rotation=90)
    ax.set_xticks(x, metric_names_plot); ax.set_ylim(0,103); ax.set_ylabel("Macro mean (%)")
    ax.set_title("Macro-mean performance across eight categories", pad=15); ax.grid(axis="y",alpha=.25); ax.legend(ncol=3,frameon=False)
    fig.savefig(REPORT_DIR/"Fig03_MacroMean_AllMetrics.png",dpi=600,bbox_inches="tight",facecolor="white"); fig.savefig(REPORT_DIR/"Fig03_MacroMean_AllMetrics.pdf",bbox_inches="tight",facecolor="white"); plt.show(); plt.close(fig)
else: print("Figure 3 unavailable: no measured metrics in CSV.")

# Figure 4: AUROC by category and model.
if "AUROC" in df_actual.columns and df_actual["AUROC"].notna().any():
    p=df_actual.pivot_table(index="Category",columns="Model",values="AUROC",aggfunc="last")
    cat_key=df_actual["Category"].astype(str).str.lower().str.replace(" ","_",regex=False); cat_map=dict(zip(cat_key,df_actual["Category"].astype(str)))
    cats=[cat_map[c] for c in category_order if c in set(cat_key)]; p=p.reindex(index=cats,columns=models_present)
    fig,ax=plt.subplots(figsize=(12,6.5),constrained_layout=True)
    for model in models_present: ax.plot(cats,p[model].to_numpy(dtype=float),marker="o",markersize=6,linewidth=2,label=model)
    ax.set_xticks(range(len(cats)),cats,rotation=22,ha="right"); ax.set_ylabel("AUROC (%)"); ax.set_ylim(0,101); ax.set_title("AUROC by category and model",pad=15); ax.grid(alpha=.25); ax.legend(ncol=3,frameon=False)
    fig.savefig(REPORT_DIR/"Fig04_AUROC_Category_Comparison.png",dpi=600,bbox_inches="tight",facecolor="white"); fig.savefig(REPORT_DIR/"Fig04_AUROC_Category_Comparison.pdf",bbox_inches="tight",facecolor="white"); plt.show(); plt.close(fig)
else: print("Figure 4 unavailable: AUROC is absent from measured CSV.")

# Figure 5: AMEM-Net AUROC gain over the strongest available baseline, category by category.
if "AUROC" in df_actual.columns and "AMEM-Net" in models_present:
    p=df_actual.pivot_table(index="Category",columns="Model",values="AUROC",aggfunc="last")
    cat_key=df_actual["Category"].astype(str).str.lower().str.replace(" ","_",regex=False); cat_map=dict(zip(cat_key,df_actual["Category"].astype(str)))
    cats=[cat_map[c] for c in category_order if c in set(cat_key)]; gains=[]; gain_cats=[]
    baselines=[m for m in models_present if m!="AMEM-Net"]
    for c in cats:
        if c not in p.index or pd.isna(p.loc[c,"AMEM-Net"]): continue
        vals=p.loc[c,baselines].dropna() if baselines else pd.Series(dtype=float)
        if len(vals): gain_cats.append(c); gains.append(float(p.loc[c,"AMEM-Net"]-vals.max()))
    if gains:
        fig,ax=plt.subplots(figsize=(12,6.5),constrained_layout=True); bars=ax.bar(gain_cats,gains); ax.axhline(0,color="black",linewidth=1)
        ax.set_ylabel("AUROC difference (percentage points)"); ax.set_title("AMEM-Net AUROC gain over strongest baseline",pad=15); ax.set_xticks(range(len(gain_cats)),gain_cats,rotation=22,ha="right"); ax.grid(axis="y",alpha=.25)
        for bar,val in zip(bars,gains): ax.annotate(f"{val:+.2f}",(bar.get_x()+bar.get_width()/2,bar.get_height()),xytext=(0,5),textcoords="offset points",ha="center",va="bottom",fontsize=10)
        fig.savefig(REPORT_DIR/"Fig05_AMEM_AUROC_Gain.png",dpi=600,bbox_inches="tight",facecolor="white"); fig.savefig(REPORT_DIR/"Fig05_AMEM_AUROC_Gain.pdf",bbox_inches="tight",facecolor="white"); plt.show(); plt.close(fig)
    else: print("Figure 5 unavailable: no category has both AMEM-Net and a measured baseline AUROC.")
else: print("Figure 5 unavailable: AMEM-Net AUROC is absent from measured CSV.")


In [27]:
# CELL 9 — FIGURES 6, 7 and 8 
# Figure 6: validation calibration
cal_cols={str(c).strip().lower().replace("_"," "):c for c in df_actual.columns}
pseudo_col=next((cal_cols[k] for k in ["validation pseudo gap","pseudo gap"] if k in cal_cols),None)
thresh_col=next((cal_cols[k] for k in ["threshold","calibration threshold"] if k in cal_cols),None)
if pseudo_col and thresh_col:
    cal=df_actual.drop_duplicates("Category").copy(); cal=cal.set_index("Category").reindex(df_actual["Category"].drop_duplicates().tolist())
    fig,axs=plt.subplots(1,2,figsize=(13,6),constrained_layout=True)
    for ax,col,title,ylabel in [(axs[0],pseudo_col,"Validation pseudo-gap","Pseudo-gap"),(axs[1],thresh_col,"Threshold","Threshold")]:
        vals=pd.to_numeric(cal[col],errors="coerce"); bars=ax.bar(np.arange(len(cal)),vals); ax.set_title(title,pad=12); ax.set_ylabel(ylabel); ax.set_xticks(range(len(cal)),cal.index,rotation=30,ha="right"); ax.grid(axis="y",alpha=.2); ax.bar_label(bars,fmt="%.4f",padding=4,fontsize=9)
    fig.savefig(REPORT_DIR/"Fig06_Validation_Calibration.png",dpi=600,bbox_inches="tight",facecolor="white"); fig.savefig(REPORT_DIR/"Fig06_Validation_Calibration.pdf",bbox_inches="tight",facecolor="white"); plt.show(); plt.close(fig)
else:
    print("Figure 6 not generated: actual CSV has no saved validation pseudo-gap and threshold columns; no values were invented.")

# Figure 7: AMEM-Net router loss from  checkpoint loss histories.
router_df = globals().get("router_loss_df", None)
if isinstance(router_df,pd.DataFrame) and {"Category","Epoch","Loss"}.issubset(router_df.columns):
    rloss=router_df.copy()
else:
    rloss=pd.DataFrame()
if not rloss.empty:
    fig,ax=plt.subplots(figsize=(12,7),constrained_layout=True)
    for category,g in rloss.groupby("Category"):
        g=g.sort_values("Epoch"); ax.plot(g["Epoch"],g["Loss"],marker="o",markersize=4,linewidth=2,label=category)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Router loss"); ax.set_title("AMEM-Net V6 router loss at logged checkpoints",pad=15); ax.grid(alpha=.25); ax.legend(ncol=2,frameon=False)
    fig.savefig(REPORT_DIR/"Fig07_AMEM_Router_Loss_Logged.png",dpi=600,bbox_inches="tight",facecolor="white"); fig.savefig(REPORT_DIR/"Fig07_AMEM_Router_Loss_Logged.pdf",bbox_inches="tight",facecolor="white"); plt.show(); plt.close(fig)
else: print("Figure 7 not generated: router-loss checkpoint history was not saved as a readable dataframe; no intermediate points were fabricated.")

# Figure 8: category-wise F1 grouped bars.
if "F1" in df_actual.columns and df_actual["F1"].notna().any():
    p=df_actual.pivot_table(index="Category",columns="Model",values="F1",aggfunc="last")
    cat_key=df_actual["Category"].astype(str).str.lower().str.replace(" ","_",regex=False); cat_map=dict(zip(cat_key,df_actual["Category"].astype(str)))
    cats=[cat_map[c] for c in category_order if c in set(cat_key)]; p=p.reindex(index=cats,columns=models_present)
    fig,ax=plt.subplots(figsize=(15,8),constrained_layout=True); x=np.arange(len(cats)); width=.15
    for i,model in enumerate(models_present):
        vals=p[model].to_numpy(dtype=float); bars=ax.bar(x+(i-(len(models_present)-1)/2)*width,vals,width,label=model); ax.bar_label(bars,fmt="%.1f",padding=3,fontsize=8,rotation=90,color="black")
    ax.set_xticks(x,cats,rotation=22,ha="right"); ax.set_ylabel("F1-score (%)"); ax.set_ylim(0,104); ax.set_title("F1-score comparison across categories",pad=15); ax.grid(axis="y",alpha=.25); ax.legend(ncol=3,frameon=False)
    fig.savefig(REPORT_DIR/"Fig08_F1_Category_Comparison.png",dpi=600,bbox_inches="tight",facecolor="white"); fig.savefig(REPORT_DIR/"Fig08_F1_Category_Comparison.pdf",bbox_inches="tight",facecolor="white"); plt.show(); plt.close(fig)
else: print("Figure 8 unavailable: F1 is absent from measured CSV.")
print("Figures and CSV tables saved to:", REPORT_DIR.resolve())


Figures and CSV tables saved to: C:\Users\chigu\AMEM_Net_IEEE_Access_Exact_User_Results
